# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH10/ch10_memory_topologies.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# About the Notebook

This notebook demonstrates how checkpointing affects memory behavior when a parent graph calls a subagent in LangGraph.

The example builds a simple billing worker that extracts information from user turns, such as duplicate charges, plan tier, and refund intent. A parent graph then calls this worker while passing only the current user message. Three execution modes are compared side by side: a fresh worker per invocation, a persistent per-thread worker, and a stateless worker without checkpointing.

The goal is to show that subagent memory depends on how the worker graph is compiled and checkpointed. Even when the parent passes only the latest turn, a checkpointer can allow the worker to retain prior messages across turns within the same thread. This makes the notebook useful for understanding memory boundaries, thread-level persistence, and subagent state handling in LangGraph-based multi-agent systems.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages


# Define worker state and node
class WorkerState(TypedDict):
    messages: Annotated[list, add_messages]


def billing_worker_node(state: WorkerState):
    human_texts = [
        m.content.lower()
        for m in state["messages"]
        if isinstance(m, HumanMessage)
    ]

    issue = "duplicate" if any("charged twice" in t for t in human_texts) else None
    tier = "pro" if any("pro plan" in t for t in human_texts) else None
    action = "refund" if any("refund" in t for t in human_texts) else None

    msg_count = len(state["messages"])
    answer = f"Msgs: {msg_count} | Known: {issue}, {tier}, {action}"
    return {"messages": [AIMessage(content=answer)]}


# Build worker graph ---
def build_worker(mode: str):
    builder = StateGraph(WorkerState)
    builder.add_node("billing_worker", billing_worker_node)
    builder.add_edge(START, "billing_worker")

    if mode == "per_invocation":
        return builder.compile()
    if mode == "per_thread":
        return builder.compile(checkpointer=True)
    if mode == "stateless":
        return builder.compile(checkpointer=False)

    raise ValueError(f"Unknown mode: {mode}")


# Parent graph passes only the current turn
class ParentState(TypedDict):
    current_input: str
    answer: str


def make_parent_app(worker_app):
    def call_worker(state: ParentState, config: RunnableConfig):
        out = worker_app.invoke(
            {"messages": [HumanMessage(content=state["current_input"])]},
            config=config,
        )
        return {"answer": out["messages"][-1].content}

    builder = StateGraph(ParentState)
    builder.add_node("call_worker", call_worker)
    builder.add_edge(START, "call_worker")
    return builder.compile(checkpointer=InMemorySaver())


# Compare modes side by side
def run_comparison():
    apps = {
        "Per-invocation (fresh start)": make_parent_app(build_worker("per_invocation")),
        "Per-thread (persistent)": make_parent_app(build_worker("per_thread")),
        "Stateless (no checkpoint)": make_parent_app(build_worker("stateless")),
    }

    turns = [
        "I was charged twice.",
        "I'm on the Pro plan.",
        "I want a refund.",
    ]

    results = []

    for label, app in apps.items():
        cfg = {"configurable": {"thread_id": f"demo-{label}"}}
        for i, turn in enumerate(turns, start=1):
            out = app.invoke({"current_input": turn}, cfg)
            results.append(
                {
                    "Mode": label,
                    "Turn": i,
                    "Input": turn,
                    "Subagent Output": out["answer"],
                }
            )

    print(f"{'MODE':<32} | {'TURN':<5} | {'SUBAGENT OUTPUT'}")
    print("-" * 95)
    for r in results:
        print(f"{r['Mode']:<32} | {r['Turn']:<5} | {r['Subagent Output']}")


run_comparison()

MODE                             | TURN  | SUBAGENT OUTPUT
-----------------------------------------------------------------------------------------------
Per-invocation (fresh start)     | 1     | Msgs: 1 | Known: duplicate, None, None
Per-invocation (fresh start)     | 2     | Msgs: 1 | Known: None, pro, None
Per-invocation (fresh start)     | 3     | Msgs: 1 | Known: None, None, refund
Per-thread (persistent)          | 1     | Msgs: 1 | Known: duplicate, None, None
Per-thread (persistent)          | 2     | Msgs: 3 | Known: duplicate, pro, None
Per-thread (persistent)          | 3     | Msgs: 5 | Known: duplicate, pro, refund
Stateless (no checkpoint)        | 1     | Msgs: 1 | Known: duplicate, None, None
Stateless (no checkpoint)        | 2     | Msgs: 1 | Known: None, pro, None
Stateless (no checkpoint)        | 3     | Msgs: 1 | Known: None, None, refund
